In [21]:
import pandas as pd
from sqlalchemy import create_engine
from config import load_db_config
import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [22]:
db_config = load_db_config()

connection_string = f"postgresql://{db_config.user}:{db_config.password}@{db_config.host}:{db_config.port}/{db_config.name}"

engine = create_engine(connection_string)

chunk_size = 10000
chunks = []

for chunk in pd.read_sql_table('games', engine, chunksize=chunk_size):
    chunks.append(chunk)

df_games = pd.concat(chunks, ignore_index=True)

df_games.head(30)

,id,igdb_id,name,slug,summary,storyline,igdb_url,cover_url,game_status,game_type,...,platforms,developers,summary_small,rating,rating_count,aggregated_rating,aggregated_rating_count,hypes,first_release_date,igdb_updated_at
0,1,1,Thief II: The Metal Age,thief-ii-the-metal-age,The ultimate thief is back! Tread softly as yo...,The game begins as Garrett continues his life ...,https://www.igdb.com/games/thief-ii-the-metal-age,https://images.igdb.com/igdb/image/upload/t_co...,NaN,0,...,[PC (Microsoft Windows)],[Looking Glass Studios],The ultimate thief is back,86.677217,152.0,90.000000,1.0,NaN,2000-03-21,2026-04-12 18:23:48
1,2,2,Thief: The Dark Project,thief-the-dark-project,Thief is a first-person stealth game that like...,NaN,https://www.igdb.com/games/thief-the-dark-project,https://images.igdb.com/igdb/image/upload/t_co...,NaN,0,...,[PC (Microsoft Windows)],[Looking Glass Studios],Thief is a first-person stealth game that like...,86.407994,174.0,70.000000,1.0,NaN,1998-11-30,2026-04-05 23:10:46
2,3,3,Thief: Deadly Shadows,thief-deadly-shadows,"In the third instalment of the Thief series, m...",NaN,https://www.igdb.com/games/thief-deadly-shadows,https://images.igdb.com/igdb/image/upload/t_co...,NaN,0,...,"[Xbox, PC (Microsoft Windows)]",[Ion Storm],"In the third instalment of the Thief series, m...",81.842990,130.0,83.000000,2.0,NaN,2004-05-25,2026-04-13 19:11:20
3,4,4,Thief,thief,There is a rising tide of fear in The City. Ha...,"Garrett, the Master Thief, steps out of the sh...",https://www.igdb.com/games/thief,https://images.igdb.com/igdb/image/upload/t_co...,NaN,0,...,"[PlayStation 3, PlayStation 4, PC (Microsoft W...",[Eidos Montréal],There is a rising tide of fear in The City,69.903972,332.0,63.642857,14.0,13.0,2014-02-25,2026-04-12 18:49:20
4,5,5,Baldur's Gate,baldur-s-gate,Baldur's Gate is a fantasy role-playing video ...,Candlekeep is an ancient fortress situated on ...,https://www.igdb.com/games/baldur-s-gate,https://images.igdb.com/igdb/image/upload/t_co...,NaN,0,...,"[Linux, PC (Microsoft Windows), iOS, Mac]","[BioWare, BioWare Edmonton]",Baldur's Gate is a fantasy role-playing video ...,85.696095,341.0,NaN,NaN,NaN,1998-12-21,2026-04-07 18:20:36
5,6,6,Baldur's Gate II: Shadows of Amn,baldurs-gate-ii-shadows-of-amn,Every World has conflict. Good and evil. Frien...,Some time after the events described in Baldur...,https://www.igdb.com/games/baldurs-gate-ii-sha...,https://images.igdb.com/igdb/image/upload/t_co...,NaN,0,...,"[Linux, PC (Microsoft Windows), Mac]","[BioWare Edmonton, BioWare, MumboJumbo]",Every World has conflict,89.847320,537.0,90.000000,1.0,NaN,2000-09-21,2026-04-10 18:18:43
6,7,7,Jagged Alliance,jagged-alliance,Jagged Alliance combines turn-based strategy a...,NaN,https://www.igdb.com/games/jagged-alliance,https://images.igdb.com/igdb/image/upload/t_co...,NaN,0,...,"[PC (Microsoft Windows), DOS]",[Madlab Software],Jagged Alliance combines turn-based strategy a...,79.067094,22.0,NaN,NaN,NaN,1995-04-15,2026-04-10 20:34:16
7,8,8,Jagged Alliance: Deadly Games,jagged-alliance-deadly-games,The enemy is on the run. One more mortar shell...,NaN,https://www.igdb.com/games/jagged-alliance-dea...,https://images.igdb.com/igdb/image/upload/t_co...,NaN,0,...,"[PC (Microsoft Windows), DOS]","[Sir-tech Software, Madlab Software]",The enemy is on the run,76.664096,5.0,NaN,NaN,NaN,1996-09-12,2026-02-25 18:39:23
8,9,9,Jagged Alliance 2,jagged-alliance-2,Jagged Alliance 2 is a perfect blend of strate...,A ruthless dictator has taken control of the t...,https://www.igdb.com/games/jagged-alliance-2,https://images.igdb.com/igdb/image/upload/t_co...,NaN,0,...,"[Linux, PC (Microsoft Windows), Mac]",[Sir-Tech Canada],Jagged Alliance 2 is a perfect blend of strate...,86.411593,49.0,NaN,NaN,NaN,1999-04-19,2026-04-12 05:18:24
9,10,11,Vampire: The Masquerade - Bloodlines,vampire-the-masquerade-bloodlines,A first- and third-person Western RPG based on...,The game plunges players into the dark and gri...,https://www.igdb.com/games/vamp

In [43]:
steam_reviews = pd.read_csv('../SteamReviewDataset1M/reviews.csv')
steam_apps = pd.read_csv('../SteamReviewDataset1M/applications.csv')

steam_reviews.head(30)

C:\Users\dan\AppData\Local\Temp\ipykernel_12032\2033476544.py:2: DtypeWarning: Columns (0: required_age) have mixed types. Specify dtype option on import or set low_memory=False.
  steam_apps = pd.read_csv('../SteamReviewDataset1M/applications.csv')


,recommendationid,appid,author_steamid,author_num_games_owned,author_num_reviews,author_playtime_forever,author_playtime_last_two_weeks,author_playtime_at_review,author_last_played,language,...,voted_up,votes_up,votes_funny,weighted_vote_score,comment_count,steam_purchase,received_for_free,written_during_early_access,created_at,updated_at
0,10000000,264220,76561198085405844,760,74,12.0,0.0,12.0,1.399060e+09,polish,...,True,0,1,0.459906,0,True,False,False,2025-09-07 12:51:00.564782+00:00,2025-09-08 00:47:55.754043+00:00
1,100001066,1006440,76561198014439859,485,234,424.0,0.0,424.0,1.632666e+09,russian,...,True,8,0,0.630770,0,True,False,False,2025-09-07 12:51:00.564782+00:00,2025-09-08 00:47:55.754043+00:00
2,100002344,320721,76561198048038590,0,385,0.0,0.0,NaN,0.000000e+00,german,...,False,1,0,0.523810,0,True,False,False,2025-09-07 12:51:00.564782+00:00,2025-09-08 00:47:55.754043+00:00
3,100002361,1604700,76561197994386273,0,3,86.0,0.0,86.0,1.632674e+09,english,...,True,3,0,0.541985,0,True,False,False,2025-09-07 12:51:00.564782+00:00,2025-09-08 00:47:55.754043+00:00
4,100002504,1338560,76561198138996331,816,26,25.0,0.0,25.0,1.632639e+09,english,...,True,1,0,0.523810,0,True,False,False,2025-09-07 12:51:00.564782+00:00,2025-09-08 00:47:55.754043+00:00
5,100002591,1418320,76561199208288640,0,1,60.0,0.0,60.0,1.631962e+09,spanish,...,True,2,0,0.545455,0,True,False,False,2025-09-07 12:51:00.564782+00:00,2025-09-08 00:47:55.754043+00:00
6,100003580,1232500,76561199089236340,0,2,311.0,0.0,261.0,1.728524e+09,japanese,...,True,2,0,0.540390,0,True,False,False,2025-09-07 12:51:00.564782+00:00,2025-09-08 00:47:55.754043+00:00
7,100003676,726870,76561198155272674,0,13,639.0,0.0,188.0,1.632773e+09,english,...,True,0,0,0.500000,0,True,False,False,2025-09-07 12:51:00.564782+00:00,2025-09-08 00:47:55.754043+00:00
8,100003804,1721670,76561198056175175,315,14,0.0,0.0,NaN,0.000000e+00,english,...,True,0,0,0.500000,0,True,False,False,2025-09-07 12:51:00.564782+00:00,2025-09-08 00:47:55.754043+00:00
9,100005000,1638870,76561199059802534,0,4,12.0,0.0,12.0,1.630245e+09,russian,...,False,0,0,0.500000,0,True,False,False,2025-09-07 12:51:00.564782+00:00,2025-09-08 00:47:55.754043+00:00


In [44]:
# True = 5, False = 1
steam_reviews['rating'] = steam_reviews['voted_up'].apply(lambda x: 5 if x else 1)

print(f'Before filtering: {len(steam_reviews)} reviews')
print(f'Users: {steam_reviews["author_steamid"].nunique()}, Games: {steam_reviews["appid"].nunique()}')

# Оставляем только активных юзеров (минимум 3 отзыва)
user_review_counts = steam_reviews.groupby('author_steamid').size()
active_users = user_review_counts[user_review_counts >= 5].index
steam_reviews = steam_reviews[steam_reviews['author_steamid'].isin(active_users)]

# Оставляем только адекватные игры (минимум 5 отзывов)
game_review_counts = steam_reviews.groupby('appid').size()
popular_games = game_review_counts[game_review_counts >= 10].index
steam_reviews = steam_reviews[steam_reviews['appid'].isin(popular_games)]

print(f'After filtering: {len(steam_reviews)} reviews')
print(f'Users: {steam_reviews["author_steamid"].nunique()}, Games: {steam_reviews["appid"].nunique()}')

Before filtering: 1048148 reviews
Users: 701884, Games: 117311
After filtering: 83912 reviews
Users: 15301, Games: 5205


In [45]:
# Привязываем Steam appid к igdb_id

if 'name' in steam_apps.columns:
    steam_apps_mapping = steam_apps[['appid', 'name']].drop_duplicates()
    game_name_to_id = dict(zip(df_games['name'].str.lower(), df_games['igdb_id']))

    def find_game_id(steam_app_id):
        app_name = steam_apps_mapping[steam_apps_mapping['appid'] == steam_app_id]['name'].values

        if len(app_name) > 0:
            return game_name_to_id.get(app_name[0].lower())

        return None

    steam_reviews['game_id'] = steam_reviews['appid'].apply(find_game_id)

    steam_reviews = steam_reviews.dropna(subset=['game_id'])
    steam_reviews['game_id'] = steam_reviews['game_id'].astype(int)

    print(f'Matched {steam_reviews["game_id"].nunique()} games')


Matched 3511 games


In [46]:
df_ratings = steam_reviews[['author_steamid', 'game_id', 'rating']].copy()
df_ratings.columns = ['user_id', 'game_id', 'rating']

df_ratings['playtime'] = steam_reviews['author_playtime_forever'].values

df_ratings['weight'] = np.log1p(df_ratings['playtime'])

len(df_ratings)

56397

In [47]:
class SVDRecommender:
    def __init__(self, n_factors=50, use_weights=False):
        self.n_factors = n_factors
        self.use_weights = use_weights
        self.user_mapping = {}
        self.game_mapping = {}
        self.game_reverse_mapping = {}
        self.mean_rating = 0
        self.U = None
        self.sigma = None
        self.Vt = None

    def fit(self, df_ratings):
        unique_users = df_ratings['user_id'].unique()
        unique_games = df_ratings['game_id'].unique()

        self.user_mapping = {uid: i for i, uid in enumerate(unique_users)}

        self.game_mapping = {gid: i for i, gid in enumerate(unique_games)}
        self.game_reverse_mapping = {i: gid for gid, i in self.game_mapping.items()}

        self.mean_rating = df_ratings['rating'].mean()

        row_ind = [self.user_mapping[uid] for uid in df_ratings['user_id']]
        col_ind = [self.game_mapping[gid] for gid in df_ratings['game_id']]

        if self.use_weights and 'weight' in df_ratings.columns:
            weights = df_ratings['weight'].fillna(1.0).values
            values = (df_ratings['rating'].values - self.mean_rating) * weights
        else:
            values = df_ratings['rating'].values - self.mean_rating

        if np.isnan(values).any() or np.isinf(values).any():
            raise ValueError('Data contains NaN or Inf')

        matrix = csr_matrix((values, (row_ind, col_ind)), shape=(len(unique_users), len(unique_games)))

        k = min(self.n_factors, min(matrix.shape) - 1)
        self.U, self.sigma, self.Vt = svds(matrix, k=k)

        index = np.argsort(self.sigma)[::-1]
        self.sigma = np.diag(self.sigma[index])
        self.U = self.U[:, index]
        self.Vt = self.Vt[index, :]

    def predict(self, user_id, game_id):
        if user_id not in self.user_mapping or game_id not in self.game_mapping:
            return self.mean_rating

        u_index = self.user_mapping[user_id]
        g_index = self.game_mapping[game_id]

        prediction = self.mean_rating + self.U[u_index, :] @ self.sigma @ self.Vt[:, g_index]

        return np.clip(prediction, 1, 5)

    def recommend_for_user(self, user_id, df_games_info, n_recommendations=10):
        if user_id not in self.user_mapping:
            return pd.DataFrame()

        user_index = self.user_mapping[user_id]

        user_vector = self.U[user_index, :] @ self.sigma
        all_predictions = self.mean_rating + user_vector @ self.Vt

        recommendations = pd.DataFrame({
            'game_id': [self.game_reverse_mapping[i] for i in range(len(all_predictions))],
            'predicted_rating': all_predictions
        })

        result = recommendations.sort_values('predicted_rating', ascending=False).merge(
            df_games_info[['igdb_id', 'name']],
            left_on='game_id',
            right_on='igdb_id'
        )

        return result.head(n_recommendations)

In [48]:
train_df, test_df = train_test_split(df_ratings, test_size=0.2, random_state=42)

recommender = SVDRecommender(n_factors=50)
recommender.fit(train_df)

In [49]:
def evaluate_model(model, test_df):
    y_true = []
    y_pred = []

    for _, row in test_df.iterrows():
        pred = model.predict(row['user_id'], row['game_id'])

        if not np.isnan(pred):
            y_true.append(row['rating'])
            y_pred.append(pred)

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    return rmse, mae


rmse, mae = evaluate_model(recommender, test_df)

print(f'RMSE = {rmse:.4f}\nMAE = {mae:.4f}')

RMSE = 1.63
MAE = 1.33


In [51]:
factors_to_test = [5, 10, 20, 50, 100]
results = []

for use_weights in [False, True]:
    text = 'With weights:\n' if use_weights else 'Pure SVD:\n'
    print(f'\nEvaluating: {text}')

    for k in factors_to_test:
        model = SVDRecommender(n_factors=k, use_weights=use_weights)
        model.fit(train_df)

        rmse, mae = evaluate_model(model, test_df)
        results.append({'method': text,'k': k, 'rmse': rmse, 'mae': mae})

        print(f'Factors: {k}\nRMSE = {rmse:.4f}\nMAE = {mae:.4f}\n')



Evaluating: Pure SVD:

Factors: 5
RMSE = 1.6269
MAE = 1.3355

Factors: 10
RMSE = 1.6269
MAE = 1.3353

Factors: 20
RMSE = 1.6264
MAE = 1.3337

Factors: 50
RMSE = 1.6258
MAE = 1.3332

Factors: 100
RMSE = 1.6254
MAE = 1.3328


Evaluating: With weights:

Factors: 5
RMSE = 1.6266
MAE = 1.3341

Factors: 10
RMSE = 1.6267
MAE = 1.3338

Factors: 20
RMSE = 1.6264
MAE = 1.3309

Factors: 50
RMSE = 1.6253
MAE = 1.3299

Factors: 100
RMSE = 1.6247
MAE = 1.3286

